# KVQuant — Harness-Integrated Implementation

This is KVQuant wired into the shared baseline harness (`_baseline_template.ipynb`)
so its results merge cleanly with every other method's `*_baseline_summary.csv`.

KVQuant is the one method that does **not** fit the per-token `forward_one_token`
hook the template uses for H2O/KIVI/SnapKV/etc. Its real implementation needs a
one-time Fisher-calibration pass and a quantizer artifact before it can even run,
and inference itself happens inside `llama_simquant.py`, invoked as a subprocess
— not a Python loop we control token-by-token in this notebook.

So instead of implementing the template's six hooks, this notebook:
1. Reuses Blocks 1-3 verbatim from the template (env setup, shared config) so
   `SHARED_SEED`, `N_EVAL_SAMPLES`, `MAX_LENGTH`, `GSM8K_MAX_NEW_TOKENS`,
   `TBT_GENERATED_TOKENS`, `GSM8K_FEWSHOT_PREFIX` are byte-identical to every
   other method's copy.
2. Clones the repo, patches the upstream KVQuant scripts (Section: Patching),
   and runs the one-time Fisher calibration + quantization.
3. Defines a single adapter function, `run_kvquant_dataset()`, that shells out
   to the patched `llama_simquant.py`, parses its stdout, and reshapes the
   result into the exact row schema `evaluate_lm_dataset`/
   `evaluate_generation_dataset` produce in the template — so Block 12
   (build table) and Block 13 (save CSVs) work completely unmodified.

Kept: the real non-uniform, Fisher-calibrated nuq4 quantization from the actual
KVQuant paper/repo (not a simplified uniform approximation) — full fidelity to
the published method, at the cost of this extra plumbing.

In [1]:
# Block 1 - Environment setup
# Mirrors the template's Block 1: pinned versions, no hardcoded tokens.

from google.colab import drive
drive.mount("/content/drive")

import os

try:
    from google.colab import userdata
    _hf_token = userdata.get("HF_TOKEN")
except Exception:
    _hf_token = os.environ.get("HF_TOKEN")

if _hf_token:
    from huggingface_hub import login
    login(token=_hf_token)
    print("Logged in to HuggingFace")
else:
    print("No HF_TOKEN found -- skipping login (fine for public models like TinyLlama)")

RESULTS_DIR = "/content/drive/MyDrive/KVQuant_Results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to: {RESULTS_DIR}")

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Block 2 - Imports, GPU check
# Same shape as the template's Block 2, trimmed to what this notebook needs
# (KVQuant's own attention-backend selection happens inside the patched
# llama_simquant.py, via FORCE_ATTN_IMPL below -- not via this notebook's
# FAST_ATTN_IMPL, which is kept here only for parity/logging).

import gc
import json as _json
import re
import shutil
import subprocess
import threading
import time
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoConfig

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA")

HAS_CUDA = torch.cuda.is_available()
if not HAS_CUDA:
    print("WARNING: No GPU detected. This will be slow.")

## Shared experiment configuration

Copied byte-for-byte from the template's Block 3 -- do not let this drift from
other methods' copies, or the cross-method comparison stops being fair.

In [ ]:
# Block 3 - Experiment settings (identical to every other method's copy)

LOCAL_MODEL_PATH = "/content/tinyllama-1.1b"
HF_MODEL_ID = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
MODEL_ID = LOCAL_MODEL_PATH if os.path.exists(LOCAL_MODEL_PATH) else HF_MODEL_ID

SHARED_SEED = 0
N_EVAL_SAMPLES = 64        # PTB/WikiText-103 window count AND GSM8K question count
MAX_LENGTH = 512           # window length for PTB/WikiText-103 (a.k.a. seqlen)
GSM8K_MAX_NEW_TOKENS = 256
USE_GSM8K_FEWSHOT = True
TBT_GENERATED_TOKENS = 20  # decode length used for TTFT/TBT probes

METHOD_NAME = "kvquant"

GSM8K_FEWSHOT_PREFIX = (
    "You are solving grade-school math word problems.\n"
    "Show the calculation briefly, then end with exactly this format:\n"
    "#### <final number>\n\n"
    "Question: Mia has 3 marbles and buys 4 more. How many marbles does she have?\n"
    "Answer: Mia has 3 + 4 = 7 marbles.\n"
    "#### 7\n\n"
    "Question: A box has 6 pencils. Sam buys 5 boxes. How many pencils does Sam buy?\n"
    "Answer: Sam buys 6 * 5 = 30 pencils.\n"
    "#### 30\n"
)

# KVQuant-specific (not part of the shared config -- other methods don't have
# an equivalent). Set to "eager" to force eager attention for a controlled
# confound check against H2O (which must use eager -- it needs real attention
# weights for heavy-hitter scoring and cannot use flash/sdpa). Leave None for
# KVQuant's normal GPU-capability-aware auto-selection.
FORCE_ATTN_IMPL = None

print("Model:", MODEL_ID)
print("Method:", METHOD_NAME)
print("MAX_LENGTH / seqlen:", MAX_LENGTH)
print("N_EVAL_SAMPLES:", N_EVAL_SAMPLES)
print("GSM8K max new tokens:", GSM8K_MAX_NEW_TOKENS)
print("GSM8K few-shot prompt:", USE_GSM8K_FEWSHOT)
print("TBT generated tokens:", TBT_GENERATED_TOKENS)
print("FORCE_ATTN_IMPL:", FORCE_ATTN_IMPL)

## Repo setup

Clone the repo, install the `gradients` (calibration) and `quant` (eval)
dependency sets. These are two separate environments on purpose: `gradients`
installs its own bundled, custom-patched transformers fork for Fisher
calibration; `quant` installs the pinned versions shared with every other
method's notebook (`transformers==4.43.4`, `accelerate==0.33.0`,
`tokenizers==0.19.1`, `huggingface_hub==0.36.2`) so environment differences
are never a confound between compression methods.

In [ ]:
# Repo setup 1/3 - Clone repo (always fresh, so patches below always apply to
# pristine upstream source rather than a possibly-already-patched copy left
# over from an earlier run in this same Colab session).
import os, shutil

if os.path.exists("/content/KVCacheCompression"):
    shutil.rmtree("/content/KVCacheCompression")
    print("Removed existing repo copy for a clean re-clone")

!git clone --recurse-submodules https://github.com/yoshikodes/KVCacheCompression.git

assert os.path.exists("/content/KVCacheCompression/KVQuant/gradients/run-fisher.py"), \
    "ERROR: run-fisher.py not found! Clone may have failed."
assert os.path.exists("/content/KVCacheCompression/KVQuant/quant/llama_simquant.py"), \
    "ERROR: llama_simquant.py not found! Clone may have failed."
print("Repo verified OK")

In [ ]:
# Repo setup 2/3 - Install gradients (Fisher calibration) dependencies
import os
assert os.path.exists("/content/KVCacheCompression/KVQuant/gradients"), \
    "ERROR: gradients dir not found. Re-run the clone cell."

%cd /content/KVCacheCompression/KVQuant/gradients
!pip install -e . -q
!pip install datasets==2.14.5 sentencepiece==0.1.99 accelerate -q -U
!pip install peft==0.6.0 -q
print("Gradients deps installed")

In [ ]:
# Repo setup 3/3 - Install quant (eval) dependencies, with flash-attn.
# Pinned to the SAME package versions as every other method's notebook, so
# environment differences are never a confound between compression methods.
import os, glob
assert os.path.exists("/content/KVCacheCompression/KVQuant/quant"), \
    "ERROR: quant dir not found. Re-run the clone cell."

%cd /content/KVCacheCompression/KVQuant/quant
!pip install -e . -q --no-deps

!pip install -q --no-deps \
  "transformers==4.43.4" \
  "accelerate==0.33.0" \
  "tokenizers==0.19.1" \
  "huggingface_hub==0.36.2"

wheels = glob.glob('/content/drive/MyDrive/flash_attn*.whl')
if wheels:
    !pip install {wheels[0]} --no-build-isolation -q
    print("Installed flash-attn from saved wheel (fast)")
else:
    !pip install flash-attn --no-build-isolation
    import glob, shutil
    fa_wheels = [w for w in glob.glob('/root/.cache/pip/wheels/**/*.whl', recursive=True) if 'flash_attn' in w]
    if fa_wheels:
        dest = f"/content/drive/MyDrive/{os.path.basename(fa_wheels[0])}"
        shutil.copy(fa_wheels[0], dest)
        print(f"Saved wheel to Drive: {dest}")

print("Quant deps installed (with flash-attn), package versions pinned to match every other method's notebook")

In [ ]:
# Helper functions - subprocess runner + output parsers + KV-cache size
# formula. These bridge KVQuant's real evaluation pipeline (a separate
# process, since it needs the calibrated quantizer) into results this
# notebook can hand to the shared Block 12/13 (table + CSV) unmodified.

RESULTS_FILE = "/content/drive/MyDrive/KVQuant_Results/all_results.json"
DATASETS = ["ptb", "wikitext103", "gsm8k"]


def check_path(path, label):
    if not os.path.exists(path):
        raise FileNotFoundError(f"ERROR: {label} not found at {path}")
    print(f"OK: {label} found at {path}")


def load_results():
    if os.path.exists(RESULTS_FILE):
        with open(RESULTS_FILE) as f:
            return _json.load(f)
    return {}


def save_result(model_name, bits, dataset, metrics):
    """Append-only historical log (across bit-widths/models over time) --
    separate from, and in addition to, the results_df/CSV output this
    notebook builds for the cross-method comparison via Block 12/13."""
    results = load_results()
    key = f"{model_name}_nuq{bits}"
    if key not in results:
        results[key] = {}
    if metrics.get("ppl") is not None:
        results[key][dataset] = metrics["ppl"]
    field_map = {
        "peak_mem_gb": "peak_memory_gb", "calculated_kv_gb": "calculated_kv_cache_gb",
        "dense_kv_gb": "dense_kv_cache_gb", "latency_sec": "latency_sec",
        "accuracy": "accuracy", "ttft_sec": "ttft_sec", "tbt_sec": "tbt_sec",
        "avg_total_eval_time_sec": "avg_total_eval_time_sec",
    }
    for src_key, dst_key in field_map.items():
        val = metrics.get(src_key)
        if val is not None:
            results[key].setdefault(dst_key, {})[dataset] = val
    if metrics.get("kv_cache_info") is not None:
        results[key].setdefault("kv_cache_theoretical", {})[dataset] = metrics["kv_cache_info"]
    results[key]["timestamp"] = datetime.now().isoformat()
    with open(RESULTS_FILE, "w") as f:
        _json.dump(results, f, indent=2)


def parse_perplexity(text):
    gsm8k_matches = re.findall(r"GSM8K Perplexity \(teacher-forced on gold answer\): ([0-9]+\.?[0-9]*)", text)
    if gsm8k_matches:
        return float(gsm8k_matches[-1])
    patterns = [
        r"Average perplexity[\s:=]+([0-9]+\.?[0-9]*)",
        r"(?:Perplexity|ppl)[\s:=]+([0-9]+\.?[0-9]*)",
        r"([0-9]+\.[0-9]+)\s*$",
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE | re.MULTILINE)
        if matches:
            return float(matches[-1])
    return None


def parse_accuracy(text):
    gsm8k_matches = re.findall(r"GSM8K Exact-Match Accuracy: ([0-9]+\.?[0-9]*)", text)
    if gsm8k_matches:
        return float(gsm8k_matches[-1])
    matches = re.findall(r"Top-1 Accuracy: ([0-9]+\.?[0-9]*)", text)
    if matches:
        return float(matches[-1])
    return None


def parse_calculated_kv_cache_gb(text):
    matches = re.findall(r"Average calculated KV cache memory: ([0-9]+\.?[0-9]*) GB", text)
    return float(matches[-1]) if matches else None


def parse_dense_kv_cache_gb(text):
    matches = re.findall(r"Reference dense KV cache memory: ([0-9]+\.?[0-9]*) GB", text)
    return float(matches[-1]) if matches else None


def parse_latency_sec(text):
    matches = re.findall(r"Eval latency: ([0-9]+\.?[0-9]*) seconds", text)
    return float(matches[-1]) if matches else None


def parse_ttft_sec(text):
    matches = re.findall(r"Average time to first token: ([0-9]+\.?[0-9]*) seconds", text)
    return float(matches[-1]) if matches else None


def parse_tbt_sec(text):
    matches = re.findall(r"Average time between tokens: ([0-9]+\.?[0-9]*) seconds", text)
    return float(matches[-1]) if matches else None


def parse_avg_total_eval_time_sec(text):
    matches = re.findall(r"Average total eval time: ([0-9]+\.?[0-9]*) seconds", text)
    return float(matches[-1]) if matches else None


def parse_max_observed_length(text):
    matches = re.findall(r"max observed length ([0-9]+) tokens", text)
    return int(matches[-1]) if matches else None


def get_gpu_memory_used_gb():
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            capture_output=True, text=True,
        )
        return float(result.stdout.strip().split("\n")[0]) / 1024
    except Exception:
        return None


def run_eval(model_path, quantizer_path, dataset):
    check_path(model_path, "model")
    check_path(quantizer_path, "quantizer")
    print(f"Running eval: {dataset} ...")

    peak_mem = [0.0]
    stop_flag = threading.Event()

    def poll_memory():
        while not stop_flag.is_set():
            mem = get_gpu_memory_used_gb()
            if mem is not None and mem > peak_mem[0]:
                peak_mem[0] = mem
            time.sleep(2)

    poll_thread = threading.Thread(target=poll_memory, daemon=True)
    poll_thread.start()

    start_time = time.time()
    subprocess_env = os.environ.copy()
    subprocess_env["N_EVAL_SAMPLES"] = str(N_EVAL_SAMPLES)
    subprocess_env["GSM8K_MAX_NEW_TOKENS"] = str(GSM8K_MAX_NEW_TOKENS)
    subprocess_env["TBT_GENERATED_TOKENS"] = str(TBT_GENERATED_TOKENS)
    subprocess_env["SHARED_SEED"] = str(SHARED_SEED)
    if FORCE_ATTN_IMPL:
        subprocess_env["FORCE_ATTN_IMPL"] = FORCE_ATTN_IMPL

    result = subprocess.run(
        [
            "python", "llama_simquant.py", model_path,
            "--abits", "4", "--nsamples", "8", "--seqlen", str(MAX_LENGTH),
            "--nuq", "--include_sparse", "--sparsity-threshold", "0.99",
            "--quantizer-path", quantizer_path, "--dataset", dataset,
        ],
        capture_output=True, text=True,
        cwd="/content/KVCacheCompression/KVQuant/quant",
        env=subprocess_env,
    )
    elapsed = time.time() - start_time

    stop_flag.set()
    poll_thread.join(timeout=3)

    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        raise RuntimeError(f"Eval failed with return code {result.returncode}")

    full_output = result.stdout + result.stderr
    metrics = {
        "ppl": parse_perplexity(full_output),
        "calculated_kv_gb": parse_calculated_kv_cache_gb(full_output),
        "dense_kv_gb": parse_dense_kv_cache_gb(full_output),
        "latency_sec": parse_latency_sec(full_output),
        "accuracy": parse_accuracy(full_output),
        "ttft_sec": parse_ttft_sec(full_output),
        "tbt_sec": parse_tbt_sec(full_output),
        "avg_total_eval_time_sec": parse_avg_total_eval_time_sec(full_output),
        "max_observed_length": parse_max_observed_length(full_output),
        "peak_mem_gb": peak_mem[0],
    }
    if metrics["ppl"] is None:
        print("Full stdout:", result.stdout)
        print("Full stderr:", result.stderr)

    print(
        f"Peak GPU memory: {peak_mem[0]:.2f} GB | Wall-clock latency: {elapsed:.1f}s | "
        f"Avg calculated KV cache: {metrics['calculated_kv_gb']} GB | Accuracy: {metrics['accuracy']} | "
        f"Avg TTFT: {metrics['ttft_sec']}s | Avg TBT: {metrics['tbt_sec']}s | "
        f"Avg total eval time: {metrics['avg_total_eval_time_sec']}s"
    )
    return metrics


def compute_kv_cache_size_gb(model_path, abits, seqlen=2048, batch_size=1, sparsity=0.0):
    """Same dense-vs-compressed KV-cache formula the template's Block 6 uses
    (theoretical_kv_cache_bytes), specialized here to also report KVQuant's
    own nuq{abits} compressed size and the resulting ratio. Reads config.json
    directly rather than loading model weights, since inference happens in
    the subprocess, not in this notebook's own process."""
    config_path = os.path.join(model_path, "config.json")
    with open(config_path) as f:
        config = _json.load(f)
    n_layers = config.get("num_hidden_layers")
    n_heads = config.get("num_attention_heads")
    n_kv_heads = config.get("num_key_value_heads", n_heads)
    hidden_size = config.get("hidden_size")
    head_dim = config.get("head_dim", hidden_size // n_heads)
    fp16_bytes = 2 * n_layers * n_kv_heads * head_dim * 2 * batch_size * seqlen
    fp16_gb = fp16_bytes / (1024 ** 3)
    quant_bytes_dense = 2 * n_layers * n_kv_heads * head_dim * (abits / 8) * batch_size * seqlen
    quant_bytes_sparse = 2 * n_layers * n_kv_heads * head_dim * 2 * batch_size * seqlen * sparsity
    quant_gb = (quant_bytes_dense + quant_bytes_sparse) / (1024 ** 3)
    compression_ratio = fp16_gb / quant_gb if quant_gb > 0 else None
    return {
        "fp16_kv_cache_gb": fp16_gb, "quantized_kv_cache_gb": quant_gb,
        "compression_ratio": compression_ratio, "n_layers": n_layers,
        "n_kv_heads": n_kv_heads, "head_dim": head_dim, "seqlen": seqlen, "abits": abits,
    }


print("Helper functions ready")

---
## Patching

Patches the upstream KVQuant scripts (cloned fresh above) to: support the
PTB/WikiText-103/GSM8K datasets, add next-token accuracy alongside
perplexity, add real wall-clock TTFT/TBT/total-eval-time + calculated
KV-cache-memory instrumentation, pick a GPU-appropriate attention backend,
and evaluate GSM8K as a real generation task (not a windowed-corpus
perplexity number). These patches are internal to the KVQuant repo's own
files and don't interact with this notebook's shared config beyond reading
the environment variables `run_eval` sets.

In [ ]:
# Patch 2.1 - run-fisher.py: load model in bfloat16
filepath = "/content/KVCacheCompression/KVQuant/gradients/run-fisher.py"
with open(filepath, "r") as f:
    content = f.read()

old = (
    "    model = transformers.AutoModelForCausalLM.from_pretrained(\n"
    "        model_args.model_name_or_path,\n"
    "        config=config,\n"
    "        cache_dir=training_args.cache_dir,\n"
    "        trust_remote_code=True\n"
    "    )"
)
new = (
    "    model = transformers.AutoModelForCausalLM.from_pretrained(\n"
    "        model_args.model_name_or_path,\n"
    "        config=config,\n"
    "        cache_dir=training_args.cache_dir,\n"
    "        trust_remote_code=True,\n"
    "        torch_dtype=__import__('torch').bfloat16\n"
    "    )"
)
if old in content:
    content = content.replace(old, new)
    with open(filepath, "w") as f:
        f.write(content)
print("run-fisher.py bfloat16 patch:", "bfloat16" in open(filepath).read())

In [ ]:
# Patch 2.2 - gradients modeling_llama.py: guard split_gpus attribute access
filepath = "/content/KVCacheCompression/KVQuant/gradients/src/transformers/models/llama/modeling_llama.py"
with open(filepath, "r") as f:
    content = f.read()

content = content.replace(
    "if self.split_gpus and idx in self.split_indices:",
    "if getattr(self, 'split_gpus', False) and idx in getattr(self, 'split_indices', []):"
)
content = content.replace(
    "if self.split_gpus and idx == len(self.layers) - 1:",
    "if getattr(self, 'split_gpus', False) and idx == len(self.layers) - 1:"
)

with open(filepath, "w") as f:
    f.write(content)

_final = open(filepath).read()
print("split_gpus patch (indices):", "getattr(self, 'split_gpus', False) and idx in getattr" in _final)
print("split_gpus patch (last layer):", "getattr(self, 'split_gpus', False) and idx == len" in _final)

In [ ]:
# Patch 2.3 - llama_simquant.py: dataset choices, accuracy metric, and the
# TTFT / TBT / avg total eval time / calculated KV cache metrics, each
# averaged across every real natural-length prompt in the eval set.
import re

filepath = "/content/KVCacheCompression/KVQuant/quant/llama_simquant.py"
with open(filepath, "r") as f:
    content = f.read()

content = content.replace(
    "choices=['wikitext2', 'c4']",
    "choices=['wikitext2', 'c4', 'ptb', 'wikitext103', 'pile']"
)

old_nll = (
    "    nlls = []\n"
    "    for i in range(nsamples):\n"
    "        hidden_states = inps[i].unsqueeze(0)\n"
    "        if norm is not None:\n"
    "            hidden_states = model.model.norm(hidden_states)\n"
    "        lm_logits = model.lm_head(hidden_states)\n"
    "        shift_logits = lm_logits[:, :-1, :].contiguous()\n"
    "        shift_labels = testenc[\n"
    "            :, (i * model.seqlen):((i + 1) * model.seqlen)\n"
    "        ][:, 1:]\n"
    "        loss_fct = nn.CrossEntropyLoss()\n"
    "        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))\n"
    "        neg_log_likelihood = loss.float() * model.seqlen\n"
    "        nlls.append(neg_log_likelihood)\n"
    "    ppl = torch.exp(torch.stack(nlls).sum() / (nsamples * model.seqlen))\n"
    "    print(ppl.item())\n"
    "    model.config.use_cache = use_cache\n"
    "    return ppl.item()"
)
new_nll = (
    "    nlls = []\n"
    "    correct = 0\n"
    "    total = 0\n"
    "    for i in range(nsamples):\n"
    "        hidden_states = inps[i].unsqueeze(0)\n"
    "        if norm is not None:\n"
    "            hidden_states = model.model.norm(hidden_states)\n"
    "        lm_logits = model.lm_head(hidden_states)\n"
    "        shift_logits = lm_logits[:, :-1, :].contiguous()\n"
    "        shift_labels = testenc[\n"
    "            :, (i * model.seqlen):((i + 1) * model.seqlen)\n"
    "        ][:, 1:]\n"
    "        loss_fct = nn.CrossEntropyLoss()\n"
    "        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))\n"
    "        neg_log_likelihood = loss.float() * model.seqlen\n"
    "        nlls.append(neg_log_likelihood)\n"
    "        preds = shift_logits.argmax(dim=-1)\n"
    "        correct += (preds == shift_labels).sum().item()\n"
    "        total += shift_labels.numel()\n"
    "    ppl = torch.exp(torch.stack(nlls).sum() / (nsamples * model.seqlen))\n"
    "    accuracy = correct / total if total > 0 else 0.0\n"
    "    print(f'Top-1 Accuracy: {accuracy:.6f}')\n"
    "    print(f'Average perplexity: {ppl.item():.6f}')\n"
    "    model.config.use_cache = use_cache\n"
    "    return ppl.item()"
)
if "Top-1 Accuracy" not in content:
    assert old_nll in content, "ERROR: could not locate llama_eval's nll block to patch (accuracy)"
    content = content.replace(old_nll, new_nll)

if "Average time to first token" not in content:
    m = re.search(r"^([ \t]*)llama_eval\(model, testloader, DEV\)[ \t]*$", content, re.MULTILINE)
    assert m, "ERROR: could not locate the llama_eval(...) call line to patch (latency)"
    indent = m.group(1)
    new_block_lines = [
        "import time as _time",
        "_eval_start = _time.time()",
        "llama_eval(model, testloader, DEV)",
        "try:",
        "    model = model.to(DEV)",
        "    torch.cuda.synchronize()",
        "    _gen_tokens = 20",
        "    _abits = getattr(args, 'abits', 4)",
        "    _sparsity_threshold = getattr(args, 'sparsity_threshold', 0.99)",
        "    _outlier_frac = max(0.0, min(1.0, 1.0 - _sparsity_threshold))",
        "    def _kv_compressed_bytes(t):",
        "        n = t.nelement()",
        "        n_outliers = max(1, int(round(n * _outlier_frac))) if _outlier_frac > 0 else 0",
        "        n_normal = n - n_outliers",
        "        b = (n_normal * _abits) / 8.0",
        "        b += n_outliers * (2 + 2)",
        "        n_channels = t.shape[-1] if t.dim() > 0 else 1",
        "        b += n_channels * 2 * 2",
        "        return b",
        "    _natural_prompts = getattr(testloader, 'natural_prompts', None) or []",
        "    assert _natural_prompts, 'No natural prompts available for this dataset (check datautils.py loader)'",
        "    print(f'Measuring latency/KV-cache over all {len(_natural_prompts)} natural prompts in this dataset...')",
        "    _ttft_list, _tbt_list, _total_list, _dense_kv_bytes_list, _compressed_kv_bytes_list, _plen_list = [], [], [], [], [], []",
        "    for _pi, _prompt_ids in enumerate(_natural_prompts):",
        "        if _pi > 0 and _pi % 100 == 0:",
        "            print(f'  ...{_pi}/{len(_natural_prompts)} prompts done')",
        "        _prompt_ids = _prompt_ids.to(DEV)",
        "        with torch.no_grad():",
        "            torch.cuda.synchronize()",
        "            _t0 = _time.time()",
        "            _out = model(_prompt_ids, use_cache=True)",
        "            torch.cuda.synchronize()",
        "            _ttft = _time.time() - _t0",
        "            _past = _out.past_key_values",
        "            _next_id = _out.logits[:, -1, :].argmax(dim=-1, keepdim=True)",
        "            _step_times = []",
        "            for _ in range(_gen_tokens - 1):",
        "                torch.cuda.synchronize()",
        "                _ts = _time.time()",
        "                _out = model(_next_id, past_key_values=_past, use_cache=True)",
        "                torch.cuda.synchronize()",
        "                _step_times.append(_time.time() - _ts)",
        "                _past = _out.past_key_values",
        "                _next_id = _out.logits[:, -1, :].argmax(dim=-1, keepdim=True)",
        "            _tbt = sum(_step_times) / len(_step_times) if _step_times else 0.0",
        "            _total = _ttft + sum(_step_times)",
        "            _dense_bytes = 0",
        "            _compressed_bytes = 0",
        "            if hasattr(_past, 'key_cache'):",
        "                for _k, _v in zip(_past.key_cache, _past.value_cache):",
        "                    if _k is not None:",
        "                        _dense_bytes += _k.element_size() * _k.nelement() + _v.element_size() * _v.nelement()",
        "                        _compressed_bytes += _kv_compressed_bytes(_k) + _kv_compressed_bytes(_v)",
        "            elif isinstance(_past, tuple):",
        "                for _layer in _past:",
        "                    if _layer is not None:",
        "                        _dense_bytes += _layer[0].element_size() * _layer[0].nelement() + _layer[1].element_size() * _layer[1].nelement()",
        "                        _compressed_bytes += _kv_compressed_bytes(_layer[0]) + _kv_compressed_bytes(_layer[1])",
        "        _ttft_list.append(_ttft)",
        "        _tbt_list.append(_tbt)",
        "        _total_list.append(_total)",
        "        _dense_kv_bytes_list.append(_dense_bytes)",
        "        _compressed_kv_bytes_list.append(_compressed_bytes)",
        "        _plen_list.append(_prompt_ids.shape[1])",
        "    _avg_ttft = sum(_ttft_list) / len(_ttft_list)",
        "    _avg_tbt = sum(_tbt_list) / len(_tbt_list)",
        "    _avg_total = sum(_total_list) / len(_total_list)",
        "    _avg_dense_kv_gb = (sum(_dense_kv_bytes_list) / len(_dense_kv_bytes_list)) / (1024**3)",
        "    _avg_kv_gb = (sum(_compressed_kv_bytes_list) / len(_compressed_kv_bytes_list)) / (1024**3)",
        "    _avg_plen = sum(_plen_list) / len(_plen_list)",
        "    _max_plen = max(_plen_list)",
        "    print(f'Average time to first token: {_avg_ttft:.4f} seconds')",
        "    print(f'Average time between tokens: {_avg_tbt:.4f} seconds')",
        "    print(f'Average total eval time: {_avg_total:.4f} seconds')",
        "    print(f'Average calculated KV cache memory: {_avg_kv_gb:.6f} GB (compressed, nuq{_abits}-{_outlier_frac*100:.1f}%, calculated from real per-prompt tensor shapes -- not a live memory measurement)')",
        "    print(f'Reference dense KV cache memory: {_avg_dense_kv_gb:.6f} GB (fp16, same real tensors, no compression)')",
        "    print(f'Measured compression ratio: {(_avg_dense_kv_gb / _avg_kv_gb):.2f}x' if _avg_kv_gb > 0 else 'Measured compression ratio: n/a')",
        "    print(f'(averaged over {len(_natural_prompts)} real prompts from this dataset, avg prompt length {_avg_plen:.1f} tokens, max observed length {_max_plen} tokens)')",
        "except Exception as _e:",
        "    print(f'Could not measure generation latency: {_e}')",
        "print(f'Eval latency: {_time.time() - _eval_start:.1f} seconds')",
    ]
    new_block = "\n".join(indent + line for line in new_block_lines)
    content = content[:m.start()] + new_block + content[m.end():]

with open(filepath, "w") as f:
    f.write(content)
_final = open(filepath).read()
print(
    "llama_simquant.py patches — datasets:", "'pile'" in _final,
    "| accuracy:", "Top-1 Accuracy" in _final,
    "| avg TTFT/TBT/total-eval-time:", "Average time to first token" in _final,
    "| avg calculated KV cache:", "Average calculated KV cache memory" in _final,
)

In [ ]:
# Patch 2.4 - llama_simquant.py: pick an attention backend the actual runtime
# GPU supports (flash-attn 2 needs Ampere+), with FORCE_ATTN_IMPL override for
# a controlled confound check against other methods.
filepath = "/content/KVCacheCompression/KVQuant/quant/llama_simquant.py"
with open(filepath, "r") as f:
    content = f.read()

old_load = "model = AutoModelForCausalLM.from_pretrained(model, config=config, trust_remote_code=True, use_flash_attention_2=True, torch_dtype=torch.half)"
new_load = (
    "import warnings as _warnings\n"
    "    import os as _attn_os\n"
    "    import torch as _torch_gpu_check\n"
    "    _cap = _torch_gpu_check.cuda.get_device_capability(0) if _torch_gpu_check.cuda.is_available() else (0, 0)\n"
    "    _attn_impl = \"flash_attention_2\" if _cap[0] >= 8 else \"sdpa\"\n"
    "    _force_attn = _attn_os.environ.get('FORCE_ATTN_IMPL')\n"
    "    if _force_attn:\n"
    "        _attn_impl = _force_attn\n"
    "    print(f'GPU compute capability: {_cap} -> using attn_implementation={_attn_impl}' + (' (forced)' if _force_attn else ''))\n"
    "    with _warnings.catch_warnings():\n"
    "        _warnings.filterwarnings(\"ignore\", message=\".*Flash Attention 2.0 with a model not initialized on GPU.*\")\n"
    "        model = AutoModelForCausalLM.from_pretrained(model, config=config, trust_remote_code=True, attn_implementation=_attn_impl, torch_dtype=torch.half)"
)
if old_load in content:
    content = content.replace(old_load, new_load)
    with open(filepath, "w") as f:
        f.write(content)
    print("llama_simquant.py attn_implementation patch applied (GPU-capability-aware)")
elif "GPU compute capability" in content:
    print("attn_implementation patch already applied, skipping")
else:
    print("WARNING: could not find the expected model-loading line to patch.")
print("Has GPU-capability check:", "GPU compute capability" in open(filepath).read())

In [ ]:
# Patch 2.5 - datautils.py: add get_ptb / get_wikitext103 / get_pile loaders
# and route to them from get_loaders. Natural prompts are token-truncated
# (not a character heuristic), capped to N_EVAL_SAMPLES read from the
# environment (shared with every other method's notebook via run_eval).
import py_compile

filepath = "/content/KVCacheCompression/KVQuant/quant/kvquant/datautils.py"
with open(filepath, "r") as f:
    lines = f.readlines()

try:
    loaders_idx = next(i for i, l in enumerate(lines) if l.strip().startswith("def get_loaders"))
except StopIteration:
    loaders_idx = len(lines)

original_lines = lines[:loaders_idx]

get_loaders_lines = []
in_loaders = False
for i, line in enumerate(lines):
    if line.startswith("def get_loaders"):
        in_loaders = True
    if in_loaders:
        get_loaders_lines.append(line)
        if i > loaders_idx and line.startswith("def ") and "get_loaders" not in line:
            get_loaders_lines = get_loaders_lines[:-1]
            break

get_loaders_src = "".join(get_loaders_lines)
old_routing = "    if 'c4' in name:"
new_routing = (
    '    if name == "ptb":\n'
    '        return get_ptb(nsamples, seed, seqlen, model)\n'
    '    if name == "wikitext103":\n'
    '        return get_wikitext103(nsamples, seed, seqlen, model)\n'
    '    if name == "pile":\n'
    '        return get_pile(nsamples, seed, seqlen, model)\n'
    "    if 'c4' in name:"
)
get_loaders_src = get_loaders_src.replace(old_routing, new_routing)

new_funcs = '''
def _build_natural_prompts(tokenizer, texts, min_tokens=3, max_tokens=256):
    """Real, individually-tokenized examples from THIS dataset, token-
    truncated (not a character heuristic), capped to N_EVAL_SAMPLES (read
    from the environment, shared with every other method's notebook) via
    deterministic seeded sampling."""
    import os, random
    n_eval = int(os.environ.get("N_EVAL_SAMPLES", "64"))
    seed = int(os.environ.get("SHARED_SEED", "0"))
    candidates = [t for t in texts if len(t.strip()) > 0]
    random.Random(seed).shuffle(candidates)
    prompts = []
    for t in candidates:
        enc = tokenizer(t, return_tensors="pt")
        if enc.input_ids.shape[1] >= min_tokens:
            prompts.append(enc.input_ids[:, :max_tokens])
        if len(prompts) >= n_eval:
            break
    return prompts


def get_ptb(nsamples, seed, seqlen, model):
    import os
    from datasets import load_dataset
    import torch, random
    from transformers import AutoTokenizer
    n_eval = int(os.environ.get("N_EVAL_SAMPLES", "64"))
    traindata = load_dataset("ptb_text_only", "penn_treebank", split="train", trust_remote_code=True)
    valdata = load_dataset("ptb_text_only", "penn_treebank", split="test", trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(model, use_fast=False)
    trainenc = tokenizer("\\n\\n".join(traindata["sentence"]), return_tensors="pt")
    valenc = tokenizer("\\n\\n".join(valdata["sentence"]), return_tensors="pt")
    assert trainenc.input_ids.shape[1] >= seqlen + 1, f"PTB train too short for seqlen={seqlen}"
    assert valenc.input_ids.shape[1] >= seqlen, f"PTB test too short for seqlen={seqlen}"
    random.seed(seed)
    trainloader = []
    for _ in range(nsamples):
        i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
        inp = trainenc.input_ids[:, i:i+seqlen]
        tar = inp.clone()
        tar[:, :-1] = -100
        trainloader.append((inp, tar))
    valenc_windowed = valenc.input_ids[:, :(n_eval * seqlen)]
    natural_prompts = _build_natural_prompts(tokenizer, valdata["sentence"])
    class TokenizerWrapper:
        def __init__(self, input_ids, natural_prompts=None):
            self.input_ids = input_ids
            self.natural_prompts = natural_prompts or []
    return trainloader, TokenizerWrapper(valenc_windowed, natural_prompts)


def get_wikitext103(nsamples, seed, seqlen, model):
    import os
    from datasets import load_dataset
    import torch, random
    from transformers import AutoTokenizer
    n_eval = int(os.environ.get("N_EVAL_SAMPLES", "64"))
    traindata = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")
    valdata = load_dataset("wikitext", "wikitext-103-raw-v1", split="test")
    tokenizer = AutoTokenizer.from_pretrained(model, use_fast=False)
    trainenc = tokenizer("\\n\\n".join(traindata["text"][:5000]), return_tensors="pt")
    valenc = tokenizer("\\n\\n".join(valdata["text"][:500]), return_tensors="pt")
    assert trainenc.input_ids.shape[1] >= seqlen + 1, f"WikiText-103 train too short for seqlen={seqlen}"
    random.seed(seed)
    trainloader = []
    for _ in range(nsamples):
        i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
        inp = trainenc.input_ids[:, i:i+seqlen]
        tar = inp.clone()
        tar[:, :-1] = -100
        trainloader.append((inp, tar))
    valenc_windowed = valenc.input_ids[:, :(n_eval * seqlen)]
    natural_prompts = _build_natural_prompts(
        tokenizer, [t for t in valdata["text"] if len(t.strip()) > 20]
    )
    class TokenizerWrapper:
        def __init__(self, input_ids, natural_prompts=None):
            self.input_ids = input_ids
            self.natural_prompts = natural_prompts or []
    return trainloader, TokenizerWrapper(valenc_windowed, natural_prompts)


def get_pile(nsamples, seed, seqlen, model):
    from datasets import load_dataset
    import torch, random
    from transformers import AutoTokenizer
    print("Loading NeelNanda/pile-10k...")
    data = load_dataset("NeelNanda/pile-10k", split="train")
    tokenizer = AutoTokenizer.from_pretrained(model, use_fast=False)
    random.seed(seed)
    MAX_RETRIES = 1000
    trainloader = []
    for _ in range(nsamples):
        for attempt in range(MAX_RETRIES):
            i = random.randint(0, len(data) - 1)
            enc = tokenizer(data[i]["text"], return_tensors="pt")
            if enc.input_ids.shape[1] >= seqlen:
                break
        else:
            raise RuntimeError(f"Could not find Pile sample with >= {seqlen} tokens")
        i = random.randint(0, enc.input_ids.shape[1] - seqlen - 1)
        inp = enc.input_ids[:, i:i+seqlen]
        tar = inp.clone()
        tar[:, :-1] = -100
        trainloader.append((inp, tar))
    random.seed(0)
    valenc = []
    for _ in range(256):
        for attempt in range(MAX_RETRIES):
            i = random.randint(0, len(data) - 1)
            tmp = tokenizer(data[i]["text"], return_tensors="pt")
            if tmp.input_ids.shape[1] >= seqlen:
                break
        else:
            raise RuntimeError(f"Could not find Pile val sample with >= {seqlen} tokens")
        i = random.randint(0, tmp.input_ids.shape[1] - seqlen - 1)
        valenc.append(tmp.input_ids[:, i:i+seqlen])
    import torch as _torch
    valenc_windowed = _torch.hstack(valenc)
    natural_prompts = _build_natural_prompts(tokenizer, [row["text"] for row in data])
    class TokenizerWrapper:
        def __init__(self, input_ids, natural_prompts=None):
            self.input_ids = input_ids
            self.natural_prompts = natural_prompts or []
    return trainloader, TokenizerWrapper(valenc_windowed, natural_prompts)
'''

final_content = "".join(original_lines) + get_loaders_src + new_funcs

with open(filepath, "w") as f:
    f.write(final_content)

try:
    py_compile.compile(filepath, doraise=True)
    print("datautils.py syntax OK | Has PTB:", "def get_ptb" in final_content,
          "| Has WT103:", "def get_wikitext103" in final_content,
          "| Has Pile:", "NeelNanda/pile-10k" in final_content,
          "| Has routing:", 'name == "ptb"' in final_content)
except py_compile.PyCompileError as e:
    print(f"datautils.py ERROR: {e}")

In [ ]:
# Patch 2.6 - datautils.py: strip trust_remote_code (not needed / causes
# issues with some dataset loaders in this environment)
file_path = "/content/KVCacheCompression/KVQuant/quant/kvquant/datautils.py"

with open(file_path, "r") as f:
    content = f.read()

old_phrase = ', trust_remote_code=True'
if old_phrase in content:
    content = content.replace(old_phrase, '')
    with open(file_path, "w") as f:
        f.write(content)
    print("Stripped all instances of trust_remote_code from datautils.py")
else:
    print("trust_remote_code phrase not found or already removed.")

In [ ]:
# Patch 2.7 - datautils.py: add GSM8K loader + route to it from get_loaders.
# Uses the SAME canonical "gsm8k"/"main" source and seeded shuffle as every
# other method's notebook, so the seeded sample selection can't diverge.
import py_compile

filepath = "/content/KVCacheCompression/KVQuant/quant/kvquant/datautils.py"
with open(filepath) as f:
    content = f.read()

new_gsm8k = '''
def _extract_gsm8k_gold_answer(answer_text):
    """GSM8K's answer field always ends with '#### <number>' -- the ground
    truth final numeric answer. Returns a float, or None if unparseable."""
    import re
    m = re.search(r"####\\s*(-?[0-9][0-9,]*\\.?[0-9]*)", answer_text)
    if not m:
        return None
    try:
        return float(m.group(1).replace(",", ""))
    except ValueError:
        return None


def get_gsm8k(nsamples, seed, seqlen, model):
    import os
    from datasets import load_dataset
    import torch, random
    from transformers import AutoTokenizer
    n_eval = int(os.environ.get("N_EVAL_SAMPLES", "64"))
    print("Loading GSM8K...")
    data = load_dataset("gsm8k", "main", split="train")
    tokenizer = AutoTokenizer.from_pretrained(model, use_fast=False)
    texts = [item["question"] + " " + item["answer"] for item in data]
    full_text = "\\n\\n".join(texts)
    trainenc = tokenizer(full_text, return_tensors="pt")
    test_data = load_dataset("gsm8k", "main", split="test")
    test_texts = [item["question"] + " " + item["answer"] for item in test_data]
    full_test = "\\n\\n".join(test_texts)
    valenc = tokenizer(full_test, return_tensors="pt")
    assert trainenc.input_ids.shape[1] >= seqlen + 1, \\
        f"GSM8K train too short ({trainenc.input_ids.shape[1]} tokens) for seqlen={seqlen}"
    assert valenc.input_ids.shape[1] >= seqlen, \\
        f"GSM8K val too short ({valenc.input_ids.shape[1]} tokens) for seqlen={seqlen}"
    random.seed(seed)
    trainloader = []
    for _ in range(nsamples):
        i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
        inp = trainenc.input_ids[:, i:i+seqlen]
        tar = inp.clone()
        tar[:, :-1] = -100
        trainloader.append((inp, tar))
    valenc_windowed = valenc.input_ids[:, :(n_eval * seqlen)]
    natural_prompts = _build_natural_prompts(tokenizer, [item["question"] for item in test_data])
    _gsm8k_seed = int(os.environ.get("SHARED_SEED", str(seed)))
    _indices = list(range(len(test_data)))
    random.Random(_gsm8k_seed).shuffle(_indices)
    qa_pairs = []
    for _i in _indices:
        item = test_data[_i]
        gold = _extract_gsm8k_gold_answer(item["answer"])
        if gold is not None:
            qa_pairs.append((item["question"], gold, item["answer"]))
        if len(qa_pairs) >= n_eval:
            break
    class TokenizerWrapper:
        def __init__(self, input_ids, natural_prompts=None, qa_pairs=None):
            self.input_ids = input_ids
            self.natural_prompts = natural_prompts or []
            self.qa_pairs = qa_pairs or []
    return trainloader, TokenizerWrapper(valenc_windowed, natural_prompts, qa_pairs)
'''

if "def get_gsm8k" not in content:
    insert_at = content.find("\ndef get_loaders")
    content = content[:insert_at] + new_gsm8k + content[insert_at:]
    print("Added get_gsm8k function")
else:
    start = content.find("def get_gsm8k")
    end = content.find("\ndef ", start + 1)
    content = content[:start] + new_gsm8k.strip("\n") + "\n" + content[end:]
    print("Replaced existing get_gsm8k function with clean version")

start = content.find("def get_loaders")
end = content.find("\ndef ", start + 1)
new_get_loaders = '''def get_loaders(
    name, nsamples=128, seed=0, seqlen=2048, model=''
):
    if 'wikitext2' in name:
        return get_wikitext2(nsamples, seed, seqlen, model)
    if 'ptb' in name:
        if 'new' in name:
            return get_ptb_new(nsamples, seed, seqlen, model)
        return get_ptb(nsamples, seed, seqlen, model)
    if name == "wikitext103":
        return get_wikitext103(nsamples, seed, seqlen, model)
    if name == "pile":
        return get_pile(nsamples, seed, seqlen, model)
    if name == "gsm8k":
        return get_gsm8k(nsamples, seed, seqlen, model)
    if 'c4' in name:
        if 'new' in name:
            return get_c4_new(nsamples, seed, seqlen, model)
        return get_c4(nsamples, seed, seqlen, model)
'''
content = content[:start] + new_get_loaders + content[end:]

with open(filepath, "w") as f:
    f.write(content)

try:
    py_compile.compile(filepath, doraise=True)
    print("datautils.py syntax OK — gsm8k loader + routing in place")
except py_compile.PyCompileError as e:
    print(f"ERROR: {e}")

In [ ]:
# Patch 2.8 - llama_simquant.py: accept gsm8k as a --dataset choice
filepath = "/content/KVCacheCompression/KVQuant/quant/llama_simquant.py"
with open(filepath, "r") as f:
    content = f.read()

old = "choices=['wikitext2', 'c4', 'ptb', 'wikitext103', 'pile']"
new = "choices=['wikitext2', 'c4', 'ptb', 'wikitext103', 'pile', 'gsm8k']"

if old in content:
    content = content.replace(old, new)
    with open(filepath, "w") as f:
        f.write(content)
    print("Patched llama_simquant.py to accept gsm8k")
elif "gsm8k" in content:
    print("gsm8k already accepted")
else:
    import re
    content = re.sub(
        r"choices=\[([^\]]+)\]",
        lambda m: m.group(0).replace("]", ", 'gsm8k']") if "gsm8k" not in m.group(0) else m.group(0),
        content, count=1
    )
    with open(filepath, "w") as f:
        f.write(content)
    print("Patched via fallback")

print("Has gsm8k:", "gsm8k" in open(filepath).read())

In [ ]:
# Patch 2.9 - llama_simquant.py: REAL GSM8K evaluation (generate an answer,
# exact-match the final number; PLUS teacher-forced perplexity on the gold
# answer given its own question), using the SAME few-shot prefix as every
# other method's notebook so GSM8K prompting is not a confound.
import re

filepath = "/content/KVCacheCompression/KVQuant/quant/llama_simquant.py"
with open(filepath, "r") as f:
    content = f.read()

if "GSM8K Exact-Match Accuracy" not in content:
    m = re.search(r"^([ \t]*)llama_eval\(model, testloader, DEV\)[ \t]*$", content, re.MULTILINE)
    assert m, "ERROR: could not locate the llama_eval(...) call line to patch (gsm8k exact-match)"
    indent = m.group(1)
    gsm8k_block_lines = [
        "try:",
        "    if getattr(args, 'dataset', None) == 'gsm8k' and getattr(testloader, 'qa_pairs', None):",
        "        import os as _gsm_os",
        "        from transformers import AutoTokenizer as _GSMTokenizer",
        "        import re as _gsm_re",
        "        _gsm_tok = _GSMTokenizer.from_pretrained(args.model, use_fast=False)",
        "        if _gsm_tok.pad_token is None:",
        "            _gsm_tok.pad_token = _gsm_tok.eos_token",
        "        model = model.to(DEV)",
        "        model.eval()",
        "        _gsm_max_new_tokens = int(_gsm_os.environ.get('GSM8K_MAX_NEW_TOKENS', '256'))",
        "        _gsm_seqlen = getattr(args, 'seqlen', 512)",
        "        _FEW_SHOT = (",
        "            'You are solving grade-school math word problems.\\n'",
        "            'Show the calculation briefly, then end with exactly this format:\\n'",
        "            '#### <final number>\\n\\n'",
        "            'Question: Mia has 3 marbles and buys 4 more. How many marbles does she have?\\n'",
        "            'Answer: Mia has 3 + 4 = 7 marbles.\\n'",
        "            '#### 7\\n\\n'",
        "            'Question: A box has 6 pencils. Sam buys 5 boxes. How many pencils does Sam buy?\\n'",
        "            'Answer: Sam buys 6 * 5 = 30 pencils.\\n'",
        "            '#### 30\\n'",
        "        )",
        "        def _extract_number(text):",
        "            m2 = _gsm_re.search(r'####\\s*(-?[0-9][0-9,]*\\.?[0-9]*)', text)",
        "            if m2:",
        "                num_str = m2.group(1)",
        "            else:",
        "                nums = _gsm_re.findall(r'-?[0-9][0-9,]*\\.?[0-9]*', text)",
        "                if not nums:",
        "                    return None",
        "                num_str = nums[-1]",
        "            num_str = num_str.replace(',', '').rstrip('.')",
        "            try:",
        "                return float(num_str)",
        "            except ValueError:",
        "                return None",
        "        _gsm_correct = 0",
        "        _gsm_total = 0",
        "        _gsm_ppl_nll_sum = 0.0",
        "        _gsm_ppl_token_count = 0",
        "        for _qi, (_gsm_q, _gsm_gt, _gsm_gold_text) in enumerate(testloader.qa_pairs):",
        "            if _qi > 0 and _qi % 100 == 0:",
        "                print(f'  ...GSM8K: {_qi}/{len(testloader.qa_pairs)} done, running accuracy={_gsm_correct/_qi:.4f}')",
        "            _prompt = _FEW_SHOT + f'\\nQuestion: {_gsm_q.strip()}\\nAnswer:'",
        "            _enc = _gsm_tok(_prompt, return_tensors='pt').to(DEV)",
        "            with torch.no_grad():",
        "                _gen_ids = model.generate(",
        "                    **_enc, max_new_tokens=_gsm_max_new_tokens, do_sample=False,",
        "                    pad_token_id=_gsm_tok.pad_token_id,",
        "                )",
        "            _gen_text = _gsm_tok.decode(_gen_ids[0][_enc['input_ids'].shape[1]:], skip_special_tokens=True)",
        "            _gen_text = _gen_text.split('Question:')[0]",
        "            _pred = _extract_number(_gen_text)",
        "            if _pred is not None and abs(_pred - _gsm_gt) < 1e-4:",
        "                _gsm_correct += 1",
        "            _gsm_total += 1",
        "            try:",
        "                _gold_number = _extract_number(_gsm_gold_text)",
        "                if _gold_number is not None:",
        "                    _gold_number_str = str(int(_gold_number)) if _gold_number == int(_gold_number) else str(_gold_number)",
        "                    _prompt_ids_list = _gsm_tok(_prompt, add_special_tokens=True)['input_ids']",
        "                    _target_ids_list = _gsm_tok(' ' + _gold_number_str, add_special_tokens=False)['input_ids']",
        "                    _max_prompt_len = max(1, _gsm_seqlen - len(_target_ids_list))",
        "                    if len(_prompt_ids_list) > _max_prompt_len:",
        "                        _prompt_ids_list = _prompt_ids_list[-_max_prompt_len:]",
        "                    _full_ids = torch.tensor([_prompt_ids_list + _target_ids_list], dtype=torch.long, device=DEV)",
        "                    with torch.no_grad():",
        "                        _ppl_out = model(_full_ids)",
        "                    _ppl_logits = _ppl_out.logits[0]",
        "                    _n_prompt = len(_prompt_ids_list)",
        "                    for _ti, _target_id in enumerate(_target_ids_list):",
        "                        _pos = _n_prompt - 1 + _ti",
        "                        if _pos < 0 or _pos >= _ppl_logits.shape[0]:",
        "                            continue",
        "                        _logp = torch.log_softmax(_ppl_logits[_pos].float(), dim=-1)",
        "                        _gsm_ppl_nll_sum += -_logp[_target_id].item()",
        "                        _gsm_ppl_token_count += 1",
        "            except Exception:",
        "                pass",
        "        _gsm_accuracy = _gsm_correct / _gsm_total if _gsm_total > 0 else 0.0",
        "        print(f'GSM8K Exact-Match Accuracy: {_gsm_accuracy:.6f}')",
        "        print(f'(evaluated via generation over {_gsm_total} GSM8K test questions -- NOT next-token prediction.)')",
        "        if _gsm_ppl_token_count > 0:",
        "            _gsm_avg_nll = _gsm_ppl_nll_sum / _gsm_ppl_token_count",
        "            _gsm_perplexity = torch.exp(torch.tensor(_gsm_avg_nll)).item()",
        "            print(f'GSM8K Perplexity (teacher-forced on gold answer): {_gsm_perplexity:.6f}')",
        "            print(f'(scored only on the gold final-answer number given its own question as context, over {_gsm_ppl_token_count} target tokens.)')",
        "except Exception as _gsm_e:",
        "    print(f'Could not run GSM8K exact-match evaluation: {_gsm_e}')",
    ]
    gsm8k_block = "\n".join(indent + line for line in gsm8k_block_lines)
    insert_at = m.end()
    content = content[:insert_at] + "\n" + gsm8k_block + content[insert_at:]
    with open(filepath, "w") as f:
        f.write(content)
    print("Added real GSM8K exact-match evaluation to llama_simquant.py")
else:
    print("GSM8K exact-match evaluation already present, skipping")

print("Has GSM8K exact-match eval:", "GSM8K Exact-Match Accuracy" in open(filepath).read())

---
## Model inferencing

Downloads TinyLlama-1.1B, runs the one-time Fisher calibration pass, then
quantizes it (nuq4, 1% outliers kept in fp16). Both of these only need to
happen once per model -- not once per dataset.

In [ ]:
# Download TinyLlama-1.1B
import shutil, os
from huggingface_hub import snapshot_download

MODEL_PATH = "/content/tinyllama-1.1b"
if os.path.exists(MODEL_PATH):
    shutil.rmtree(MODEL_PATH)
    print("Deleted existing folder for clean download")

snapshot_download(repo_id="TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T", local_dir=MODEL_PATH)

files = os.listdir(MODEL_PATH)
assert any(f.endswith(".safetensors") or f.endswith(".bin") for f in files), \
    "ERROR: No model weights found. Download may have failed."
print(f"Model verified OK: {MODEL_PATH}")

In [ ]:
# Fisher calibration (gradients)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

check_path("/content/tinyllama-1.1b", "TinyLlama-1.1B model")

%cd /content/KVCacheCompression/KVQuant/gradients

# PYTHONPATH is set explicitly so this subprocess imports the gradients
# folder's custom-patched transformers fork (needed for Fisher calibration --
# it defines LlamaModel.set_devices(), which Patch 2.2 touches) instead of
# the plain transformers==4.43.4 the next (Quantize) cell installs, which
# would otherwise silently shadow it in site-packages.
!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=/content/KVCacheCompression/KVQuant/gradients/src python run-fisher.py --model_name_or_path /content/tinyllama-1.1b --output_dir /content/fisher-tinyllama-1.1b --dataset wikitext2 --seqlen 256 --maxseqlen 512 --num_examples 8

check_path("/content/fisher-tinyllama-1.1b", "Fisher output")
print("Fisher done for TinyLlama-1.1B")

In [ ]:
# Quantize (nuq4, 1% outliers)
import sys, os

sys.path = [p for p in sys.path if 'gradients' not in p]

check_path("/content/tinyllama-1.1b", "TinyLlama-1.1B model")
check_path("/content/fisher-tinyllama-1.1b", "Fisher output for TinyLlama-1.1B")

%cd /content/KVCacheCompression/KVQuant/quant

!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=/content/KVCacheCompression/KVQuant/quant python llama_simquant.py /content/tinyllama-1.1b --abits 4 --nsamples 8 --seqlen 512 --nuq --fisher /content/fisher-tinyllama-1.1b --quantize --include_sparse --sparsity-threshold 0.99 --quantizer-path /content/quantizers_tinyllama-1.1b.pickle

check_path("/content/quantizers_tinyllama-1.1b.pickle", "Quantizer pickle for TinyLlama-1.1B")
print("Quantization done for TinyLlama-1.1B")

---
## Adapter — replaces the template's Block 7 + Blocks 8-10

KVQuant can't implement `forward_one_token`; instead, `run_kvquant_dataset()`
below shells out to the patched `llama_simquant.py` (via `run_eval`, defined
in the helper functions cell), parses its output, and returns a dict shaped
exactly like the rows `evaluate_lm_dataset`/`evaluate_generation_dataset`
produce in every other method's notebook. `kv_cache_accounting` is set to
`"calculated"` -- KVQuant's simulated-quantization pipeline never physically
shrinks the live cache tensors, so its memory number is a formula applied to
real shapes, not a live measurement (see the template's Block 6 for the same
distinction on the eviction-method side).

In [ ]:
# Adapter - run KVQuant on PTB, WikiText-103, GSM8K and build results_df in
# the SAME schema the per-token-hook methods produce.

MODEL_PATH = "/content/tinyllama-1.1b"
QUANTIZER_PATH = "/content/quantizers_tinyllama-1.1b.pickle"
KV_CACHE_ACCOUNTING = "calculated"  # simulated quantization -- never physically shrinks the live cache

DATASET_KEY_MAP = {"PTB": "ptb", "WikiText-103": "wikitext103", "GSM8K": "gsm8k"}
DATASET_LABEL_MAP = {v: k for k, v in DATASET_KEY_MAP.items()}


def run_kvquant_dataset(dataset_name):
    dataset_key = DATASET_KEY_MAP[dataset_name]
    metrics = run_eval(MODEL_PATH, QUANTIZER_PATH, dataset_key)

    max_len = metrics.get("max_observed_length") or MAX_LENGTH
    kv_info = compute_kv_cache_size_gb(MODEL_PATH, abits=4, seqlen=max_len, sparsity=0.01)

    if metrics.get("ppl"):
        save_result("tinyllama-1.1b", 4, dataset_key, metrics)
    else:
        print(f"WARNING: could not parse perplexity for {dataset_name}. Check logs above.")

    return {
        "dataset": dataset_name,
        "dataset_type": "generation" if dataset_name == "GSM8K" else "language_modeling",
        "method": METHOD_NAME,
        "samples": N_EVAL_SAMPLES,
        "tokens": None,
        "perplexity": metrics["ppl"],
        "accuracy": metrics["accuracy"],
        "accuracy_type": "gsm8k_numeric_exact_match" if dataset_name == "GSM8K" else "next_token_accuracy",
        "reference_dense_kv_cache_gb": kv_info["fp16_kv_cache_gb"],
        "measured_kv_cache_gb": metrics.get("dense_kv_gb"),
        "calculated_kv_cache_gb": metrics.get("calculated_kv_gb"),
        "reported_kv_cache_gb": metrics.get("calculated_kv_gb"),
        "kv_cache_accounting": KV_CACHE_ACCOUNTING,
        "max_cache_tokens": max_len,
        "ttft_sec": metrics.get("ttft_sec"),
        "tbt_sec": metrics.get("tbt_sec"),
        "ttft_tbt_n_prompts": None,
        "total_eval_time_sec": metrics.get("avg_total_eval_time_sec"),
        "ms_per_token": None,
        "tokens_per_sec": None,
    }


print("=" * 80)
print(f"Running {METHOD_NAME}")
print("=" * 80)

summary_rows = []
for dataset_name in ["PTB", "WikiText-103", "GSM8K"]:
    print()
    print("-" * 80)
    print("Evaluating dataset:", dataset_name)
    print("-" * 80)
    %cd /content/KVCacheCompression/KVQuant/quant
    row = run_kvquant_dataset(dataset_name)
    summary_rows.append(row)
    print(row)

results_df = pd.DataFrame(summary_rows)
samples_df = pd.DataFrame()  # KVQuant's subprocess pipeline doesn't currently emit per-sample rows

print()
print("Final results_df datasets:", results_df["dataset"].tolist() if not results_df.empty else [])
display(results_df)

In [ ]:
# Block 12 - Build the baseline table
# Unchanged from the template -- reads only results_df's columns, so it
# works the same whether the rows came from a per-token loop or (as here) a
# subprocess adapter.

DATASET_ORDER = ["PTB", "WikiText-103", "GSM8K"]
METRIC_ROWS = [
    "Perplexity", "Accuracy", "Reference Dense Memory (GB)", "Reported Memory (GB)",
    "TTFT (s)", "TBT (s)", "Total eval time (s)",
]


def _get_metric_value(df, dataset, metric):
    rows = df[df["dataset"] == dataset]
    if rows.empty:
        return "N/A"
    r = rows.iloc[0]
    mapping = {
        "Perplexity": "perplexity", "Accuracy": "accuracy",
        "Reference Dense Memory (GB)": "reference_dense_kv_cache_gb",
        "Reported Memory (GB)": "reported_kv_cache_gb",
        "TTFT (s)": "ttft_sec", "TBT (s)": "tbt_sec", "Total eval time (s)": "total_eval_time_sec",
    }
    v = r.get(mapping[metric], float("nan"))
    if pd.isna(v):
        return "N/A"
    if metric == "Accuracy":
        return f"{float(v):.4f}"
    if metric in {"Reference Dense Memory (GB)", "Reported Memory (GB)"}:
        return f"{float(v):.6f}"
    return f"{float(v):.4f}"


table_rows = [{"Method": METHOD_NAME, **{ds: "" for ds in DATASET_ORDER}}]
for metric in METRIC_ROWS:
    table_rows.append({"Method": metric, **{ds: _get_metric_value(results_df, ds, metric) for ds in DATASET_ORDER}})

baseline_table_df = pd.DataFrame(table_rows, columns=["Method"] + DATASET_ORDER)

accounting = results_df["kv_cache_accounting"].iloc[0] if not results_df.empty else "n/a"
print(f"{METHOD_NAME} baseline table (memory accounting for this method: {accounting}):")
display(baseline_table_df)
display(results_df)

In [ ]:
# Block 13 - Save results, namespaced by method (unchanged from the template)

summary_path = f"/content/{METHOD_NAME}_baseline_summary.csv"
samples_path = f"/content/{METHOD_NAME}_baseline_samples.csv"
table_path = f"/content/{METHOD_NAME}_baseline_table.csv"

results_df.to_csv(summary_path, index=False)
samples_df.to_csv(samples_path, index=False)
baseline_table_df.to_csv(table_path, index=False)

print("Saved summary:", summary_path)
print("Saved samples:", samples_path)
print("Saved table:", table_path)

## Notes

- This notebook's `results_df`/CSV output uses the exact same column schema
  as every per-token-hook method notebook built from `_baseline_template.ipynb`
  (H2O, and future KIVI/SnapKV/RocketKV/ZipCache/FastGen copies) -- so all of
  them can be concatenated by `dataset`/`method` into one cross-method table.
- `samples_df` is intentionally empty here: the subprocess pipeline doesn't
  currently emit a per-example row for every PTB window / GSM8K question the
  way the per-token loop does. If you want per-sample granularity later,
  Patch 2.9's GSM8K loop is the place to add per-question print statements
  that a new parser can pick up.
- Attention backend: KVQuant auto-selects flash/sdpa based on the runtime
  GPU; set `FORCE_ATTN_IMPL = "eager"` in Block 3 to run a controlled
  confound check against a method (like H2O) that structurally requires
  eager attention.
- Calibration: KVQuant calibrates once on WikiText2 and is evaluated
  cross-domain on PTB/WikiText-103/GSM8K; per-token eviction methods have no
  separate calibration phase. Call this out as a genuine structural
  difference in your write-up, not something to force into alignment.
- Before trusting a comparison against another method's CSV: confirm both
  runs used the same actual Colab GPU type -- nothing here pins that for you.